# OrderBot

We can automate the collection of user prompts and assistant responses to build an  OrderBot. The OrderBot will take orders at a pizza restaurant.

The chat window is built with **Gradio**, which is not part of AI500Env. The next cell installs it, with exact versions that work with AI500Env. Run it **once**, then restart the kernel (Kernel > Restart) and run the notebook from the top. You only need to do this once: the installed packages stay in your account, so notebook 02B and the ChatBot assignment can use them too.

Run the cells in order. The Gradio cell prints a public `https://….gradio.live` link: open it in a new browser tab to chat (see the note above the Gradio cell). If the chatbot stops responding, click **New order**, or restart the kernel and run the cells again.

In [ ]:
# Run once, then restart the kernel (Kernel > Restart). Installs Gradio with versions that work with AI500Env.
%pip install gradio==6.15.1 gradio_client==2.5.0 fastapi==0.128.1 starlette==0.50.0 annotated-doc==0.0.6 groovy==0.1.2 hf-gradio==0.4.1 pydub==0.25.1 python-multipart==0.0.33 safehttpx==0.1.7 semantic-version==2.10.0 tomlkit==0.14.0

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
import os

# define the environment path. it is at the parent folder
env_path ="../.env"

load_dotenv(dotenv_path=env_path, override=True)

client = OpenAI()

# select a model.
# You can try different models to see which one works the best.

my_model="gpt-5.4-mini"

def get_completion(prompt, model=my_model):
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0, # we want the model to give us the best response, no variation.
    )
    return response.choices[0].message.content

def get_completion_from_messages(messages, model=my_model, temperature=0):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature, # this is the degree of randomness of the model's output
    )
    return response.choices[0].message.content

context = [ {'role':'system', 'content':"""
You are OrderBot, an automated service to collect orders for a pizza restaurant.
You first greet the customer, then collect the order, 
and then ask if it's a pickup or delivery.
You wait to collect the entire order, then summarize it and check for a final
time if the customer wants to add anything else.
If it's a delivery, you ask for an address.
tell customer the final price of the order at the end, please make sure you add cost for extra toppings.
if it is a delivery, add a 5 dollar delivery fee.
Finally you collect the payment.
Make sure to clarify all options, extras and sizes to uniquely
identify the item from the menu.
You respond in a short, very conversational friendly style.
The menu includes
pepperoni pizza  12.95, 10.00, 7.00
cheese pizza   10.95, 9.25, 6.50
eggplant pizza   11.95, 9.75, 6.75
fries 4.50, 3.50
greek salad 7.25
Toppings:
extra cheese 2.00,
mushrooms 1.50
sausage 3.00
canadian bacon 3.50
AI sauce 1.50
peppers 1.00
Drinks:
coke 3.00, 2.00, 1.00
sprite 3.00, 2.00, 1.00
bottled water 5.00
"""} ]  # accumulate messages

## Chat with OrderBot (Gradio)

Type your order and click **Chat!** (or press Enter). **New order** clears the chat and starts over. Run the summary cells below **before** you click it; if you forget, the summary uses the order you just finished.

When you are done, click **Quit** to stop the app, then run the summary cells below.

> **Open the public link.** The cell prints two links. Click the `https://….gradio.live` link (next to *Running on public URL*). The local `http://127.0.0.1:…` link points to the HPC server itself, not to your computer, so it gives an error in your browser. The chat window shown below the cell may also stay blank on the HPC; use the public link.

> **Keep your app link private.** The `https://….gradio.live` link works for anyone who has it, for about a week, and every message is charged to your API key. Do not post it in discussions or screenshots, and click **Quit** when you are done: it stops the app and the link.

In [ ]:
import gradio as gr  # GUI
import threading

START = context.copy()   # the system prompt, which every new order starts with
last_order = START        # the last finished order, so the summary still works after New order

def respond(user_message, chat_history):
    # add the customer's message to the conversation, get the reply, and show both in the chat window
    context.append({'role':'user', 'content':f"{user_message}"})
    response = get_completion_from_messages(context)
    context.append({'role':'assistant', 'content':f"{response}"})
    chat_history = chat_history + [{"role": "user", "content": user_message},
                                   {"role": "assistant", "content": response}]
    return "", chat_history

def new_order():
    # save the finished order for the summary, then start over with only the system prompt
    global last_order
    if len(context) > len(START):
        last_order = context.copy()
    context[:] = START
    return "", []

def quit_app(chat_history):
    # stop the app; the conversation stays in context, so the summary cells below still work
    threading.Timer(1, demo.close).start()   # close the server a second after the goodbye message shows
    chat_history = chat_history + [{"role": "assistant",
                                    "content": "OrderBot has stopped. Go back to the notebook and run the summary cells."}]
    off = gr.Button(interactive=False)
    return gr.Textbox(value="", interactive=False), chat_history, off, off, off

with gr.Blocks(title="OrderBot") as demo:
    gr.Markdown("# 🍕 OrderBot")
    chatbot = gr.Chatbot(height=500)
    msg = gr.Textbox(value="Hi", placeholder="Enter text here…", label="Your message")
    with gr.Row():
        submit_btn = gr.Button("Chat!")
        new_order_btn = gr.Button("New order")
        quit_btn = gr.Button("Quit")

    msg.submit(respond, inputs=[msg, chatbot], outputs=[msg, chatbot])
    submit_btn.click(respond, inputs=[msg, chatbot], outputs=[msg, chatbot])
    new_order_btn.click(new_order, outputs=[msg, chatbot])
    quit_btn.click(quit_app, inputs=[chatbot], outputs=[msg, chatbot, submit_btn, new_order_btn, quit_btn])

demo.launch(share=True)

## Look at the conversation and summarize the order

After you finish an order (before you click **New order**), the cells below show the full conversation the model saw, and ask the model for a JSON summary of the order.

In [ ]:
# the conversation for the current order, or the last finished order if you clicked New order
order = context if len(context) > len(START) else last_order
order

In [ ]:
messages = order.copy()
messages.append(
{'role':'system', 'content':'create a json summary of the previous food order. Itemize the price for each item\
 The fields should be 1) pizza, include size 2) list of toppings 3) list of drinks, include size   4) list of sides include size  5)total price '},    
)

response = get_completion_from_messages(messages, temperature=0)
print(response)

## Try another model (optional)

The OrderBot does the price math **itself**, and language models are not calculators. The cell below sends a larger
order, without the chat window, and prints the bot's total. The correct total is **$40.00**:

| Item | Price |
|---|---|
| 2 medium cheese pizzas, each with extra cheese and sausage: 2 × (9.25 + 2.00 + 3.00) | 28.50 |
| Large fries | 4.50 |
| 2 small sprites: 2 × 1.00 | 2.00 |
| Delivery fee | 5.00 |
| **Total** | **40.00** |

1. Run the cell two or three times with `gpt-5.4-mini`. Is the total right each time?
2. Change `test_model` to `"gpt-5.4"` (a larger model) and run it again. Then try `"gpt-4.1-mini"`.
3. Think about it: does a larger model solve the problem? It costs more and is slower, and it still cannot *guarantee*
   a correct total. Notebook **02B** solves the problem differently: the model asks our code to do the math.

> Use the models listed above. Some newer models (for example `gpt-5.5` or `gpt-5-nano`) give an error here, because
> this notebook sets `temperature=0` and those models do not accept that setting.

In [ ]:
# Optional: test a model on a larger order (no chat window needed)
test_model = "gpt-5.4-mini"   # then try "gpt-5.4" or "gpt-4.1-mini"

customer_messages = ["Hi",
                     "Two medium cheese pizzas, each with extra cheese and sausage, a large fries, and two small sprites.",
                     "Delivery to 45 Elm Street.",
                     "That's everything. What's my total?"]

conversation = [context[0]]   # start with the OrderBot system prompt only
for text in customer_messages:
    conversation.append({'role':'user', 'content':text})
    reply = get_completion_from_messages(conversation, model=test_model)
    conversation.append({'role':'assistant', 'content':reply})
    print("CUSTOMER:", text)
    print("BOT     :", reply)
    print()

print(f"Model: {test_model}.  The correct total is $40.00.")